In [ ]:
"""
Poem Similarity Calculator — Production Quality
=================================================
Uses the best available embedding method to compute semantic similarity
between Arabic poems, with quality-ranked fallbacks.

Quality ranking (best → worst):
  1. OpenAI text-embedding-3-large  (best for Arabic, ~$0.13/1M tokens)
  2. Cohere embed-v3 multilingual   (excellent Arabic support)
  3. Voyage AI voyage-multilingual-2 (strong multilingual)
  4. Local: intfloat/multilingual-e5-large (free, very good)
  5. Local: paraphrase-multilingual-MiniLM-L12-v2 (free, good)
  6. TF-IDF cosine (free, weakest — word overlap only)

Install requirements based on your chosen method:
  pip install pandas numpy scikit-learn openpyxl tqdm

  # For Option 1 (OpenAI):
  pip install openai

  # For Option 4/5 (Local embeddings):
  pip install sentence-transformers torch
"""
%pip install pandas numpy scikit-learn openpyxl tqdm openai
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json
import re
import time
import warnings
warnings.filterwarnings('ignore')


# ══════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════
INPUT_FILE = ".\CSV\Diwan-Hamdan-WIP - Full_poems.csv"
OUTPUT_FILE = "poems_with_similarity.csv"
TOP_N = 10
MIN_SCORE = 0.1       # Set to e.g. 0.3 to filter weak matches

# Choose your embedding method: "openai", "cohere", "voyage", "local-e5", "local-minilm", "tfidf"
EMBEDDING_METHOD = "openai"

# API keys (only needed for cloud methods)
OPENAI_API_KEY = "..."       # For "openai"
COHERE_API_KEY = "..."          # For "cohere"
VOYAGE_API_KEY = "..."          # For "voyage"

# Which text columns to combine for similarity
TEXT_COLUMNS = {
    "Poem_line_cleaned": 3,      # Poem body — most important
    "Extended_Summary": 2,       # Rich context
    "Summary": 1,                # Short summary
    "Titlecleaned": 1,           # Title
}

# Batch size for API calls (to avoid rate limits)
API_BATCH_SIZE = 50
API_DELAY = 0.5  # seconds between batches


# ══════════════════════════════════════════════
# TEXT PREPARATION
# ══════════════════════════════════════════════
def clean_arabic(text):
    """Clean and normalize Arabic text."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)  # remove tashkeel
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'["""\'«»]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_texts(df):
    """
    Build a single text per poem for embedding.
    For LLM embeddings, we structure it semantically rather than repeating.
    """
    texts = []
    for _, row in df.iterrows():
        parts = []

        title = clean_arabic(row.get('Titlecleaned', ''))
        if title:
            parts.append(f"العنوان: {title}")

        summary = clean_arabic(row.get('Summary', ''))
        if summary:
            parts.append(f"الملخص: {summary}")

        ext_summary = clean_arabic(row.get('Extended_Summary', ''))
        if ext_summary:
            parts.append(f"التحليل: {ext_summary}")

        poem = clean_arabic(row.get('Poem_line_cleaned', ''))
        if poem:
            parts.append(f"القصيده: {poem}")

        texts.append("\n".join(parts))
    return texts


# ══════════════════════════════════════════════
# EMBEDDING METHODS
# ══════════════════════════════════════════════

# ─── 1. OpenAI (BEST for Arabic) ───
def embed_openai(texts):
    """
    OpenAI text-embedding-3-large
    - 3072 dimensions, best-in-class multilingual
    - ~$0.13 per 1M tokens
    - For 300 poems ≈ $0.02-0.05
    """
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    all_embeddings = []
    for i in range(0, len(texts), API_BATCH_SIZE):
        batch = texts[i:i + API_BATCH_SIZE]
        print(f"  OpenAI batch {i // API_BATCH_SIZE + 1}/{(len(texts) - 1) // API_BATCH_SIZE + 1}")

        response = client.embeddings.create(
            model="text-embedding-3-large",
            input=batch
        )
        batch_emb = [item.embedding for item in response.data]
        all_embeddings.extend(batch_emb)

        if i + API_BATCH_SIZE < len(texts):
            time.sleep(API_DELAY)

    return np.array(all_embeddings)


# ─── 2. Cohere (Excellent multilingual) ───
def embed_cohere(texts):
    """
    Cohere embed-multilingual-v3.0
    - 1024 dimensions, excellent Arabic
    - Free tier: 100 calls/min
    """
    import cohere
    co = cohere.Client(COHERE_API_KEY)

    all_embeddings = []
    for i in range(0, len(texts), 96):  # Cohere max batch = 96
        batch = texts[i:i + 96]
        print(f"  Cohere batch {i // 96 + 1}/{(len(texts) - 1) // 96 + 1}")

        response = co.embed(
            texts=batch,
            model="embed-multilingual-v3.0",
            input_type="search_document"
        )
        all_embeddings.extend(response.embeddings)

        if i + 96 < len(texts):
            time.sleep(API_DELAY)

    return np.array(all_embeddings)


# ─── 3. Voyage AI ───
def embed_voyage(texts):
    """
    Voyage voyage-multilingual-2
    - Strong multilingual embeddings
    """
    import voyageai
    vo = voyageai.Client(api_key=VOYAGE_API_KEY)

    all_embeddings = []
    for i in range(0, len(texts), API_BATCH_SIZE):
        batch = texts[i:i + API_BATCH_SIZE]
        print(f"  Voyage batch {i // API_BATCH_SIZE + 1}/{(len(texts) - 1) // API_BATCH_SIZE + 1}")

        result = vo.embed(batch, model="voyage-multilingual-2", input_type="document")
        all_embeddings.extend(result.embeddings)

        if i + API_BATCH_SIZE < len(texts):
            time.sleep(API_DELAY)

    return np.array(all_embeddings)


# ─── 4. Local: E5-large (best free option) ───
def embed_local_e5(texts):
    """
    intfloat/multilingual-e5-large
    - 1024 dimensions, very strong multilingual
    - Free, runs locally, ~1.2GB model
    - Requires: pip install sentence-transformers torch
    """
    from sentence_transformers import SentenceTransformer

    print("  Loading multilingual-e5-large (first run downloads ~1.2GB)...")
    model = SentenceTransformer("intfloat/multilingual-e5-large")

    # E5 models need "passage: " prefix
    prefixed = [f"passage: {t}" for t in texts]

    print("  Encoding poems...")
    embeddings = model.encode(prefixed, show_progress_bar=True, batch_size=16)
    return np.array(embeddings)


# ─── 5. Local: MiniLM (lightweight free option) ───
def embed_local_minilm(texts):
    """
    paraphrase-multilingual-MiniLM-L12-v2
    - 384 dimensions, decent multilingual
    - Free, runs locally, ~470MB model
    """
    from sentence_transformers import SentenceTransformer

    print("  Loading MiniLM (first run downloads ~470MB)...")
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    print("  Encoding poems...")
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
    return np.array(embeddings)


# ─── 6. TF-IDF fallback ───
def embed_tfidf(texts):
    """TF-IDF — word overlap only, weakest but zero dependencies."""
    from sklearn.feature_extraction.text import TfidfVectorizer

    print("  Building TF-IDF matrix...")
    vectorizer = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True,
    )
    return vectorizer.fit_transform(texts).toarray()


# ══════════════════════════════════════════════
# SIMILARITY COMPUTATION
# ══════════════════════════════════════════════
def get_embeddings(texts, method):
    """Route to the chosen embedding method."""
    print(f"\nGenerating embeddings using: {method}")

    methods = {
        "openai": embed_openai,
        "cohere": embed_cohere,
        "voyage": embed_voyage,
        "local-e5": embed_local_e5,
        "local-minilm": embed_local_minilm,
        "tfidf": embed_tfidf,
    }

    if method not in methods:
        raise ValueError(f"Unknown method '{method}'. Choose from: {list(methods.keys())}")

    embeddings = methods[method](texts)
    print(f"  Embedding shape: {embeddings.shape}")
    return embeddings


def generate_similar_poems(df, sim_matrix, top_n=5, min_score=0.0):
    """
    For each poem, find top_n most similar poems and format as JSON:
      [{"poem_id": 7, "rank": 1, "similarity_score": 0.4523}, ...]
    """
    poem_ids = df['poem_id'].values
    similar_col = []

    for i in range(len(df)):
        scores = sim_matrix[i]
        pairs = [(j, scores[j]) for j in range(len(scores)) if j != i]
        pairs.sort(key=lambda x: x[1], reverse=True)
        top_pairs = [(j, s) for j, s in pairs if s >= min_score][:top_n]

        result = []
        for rank, (j, score) in enumerate(top_pairs, start=1):
            result.append({
                "poem_id": int(poem_ids[j]),
                "rank": rank,
                "similarity_score": round(float(score), 4)
            })

        similar_col.append(json.dumps(result, ensure_ascii=False))

    return similar_col


# ══════════════════════════════════════════════
# OPTIONAL: Save/Load embeddings cache
# ══════════════════════════════════════════════
def save_embeddings(embeddings, filepath="embeddings_cache.npy"):
    """Cache embeddings to avoid recomputing."""
    np.save(filepath, embeddings)
    print(f"  Embeddings cached to '{filepath}'")


def load_embeddings(filepath="embeddings_cache.npy"):
    """Load cached embeddings if available."""
    try:
        embeddings = np.load(filepath)
        print(f"  Loaded cached embeddings from '{filepath}' — shape: {embeddings.shape}")
        return embeddings
    except FileNotFoundError:
        return None


# ══════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════
def main():
    # 1. Load data
    if INPUT_FILE.endswith('.csv'):
        df = pd.read_csv(INPUT_FILE)
    else:
        df = pd.read_excel(INPUT_FILE)

    print(f"Loaded {len(df)} poems from '{INPUT_FILE}'")

    if 'poem_id' not in df.columns:
        df['poem_id'] = df.index + 1

    # 2. Build texts
    print("\nPreparing text...")
    texts = build_texts(df)

    # 3. Get embeddings (try cache first)
    cache_file = f"embeddings_cache_{EMBEDDING_METHOD}.npy"
    embeddings = load_embeddings(cache_file)

    if embeddings is None or len(embeddings) != len(df):
        embeddings = get_embeddings(texts, EMBEDDING_METHOD)
        save_embeddings(embeddings, cache_file)

    # 4. Compute similarity
    print("\nComputing cosine similarity...")
    sim_matrix = cosine_similarity(embeddings)

    # 5. Generate similar_poems column
    print(f"Finding top {TOP_N} similar poems (min_score={MIN_SCORE})...")
    df['similar_poems'] = generate_similar_poems(df, sim_matrix, top_n=TOP_N, min_score=MIN_SCORE)

    # 6. Save
    if OUTPUT_FILE.endswith('.csv'):
        df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    else:
        df.to_excel(OUTPUT_FILE, index=False)

    print(f"\nSaved to '{OUTPUT_FILE}'")

    # 7. Preview
    print("\n" + "=" * 70)
    print(f"PREVIEW (method: {EMBEDDING_METHOD})")
    print("=" * 70)
    for _, row in df.head(5).iterrows():
        print(f"\n  Poem {row['poem_id']}: {row.get('Title_raw', 'N/A')[:40]}")
        print(f"  related_poems:  {row.get('related_poems', 'N/A')}")
        similar = json.loads(row['similar_poems'])
        for s in similar:
            print(f"    -> poem {s['poem_id']:>4}  rank:{s['rank']}  score:{s['similarity_score']:.4f}")

    # 8. Score distribution stats
    all_scores = []
    for val in df['similar_poems']:
        for s in json.loads(val):
            all_scores.append(s['similarity_score'])

    if all_scores:
        print(f"\n{'=' * 70}")
        print("SIMILARITY SCORE DISTRIBUTION")
        print(f"{'=' * 70}")
        print(f"  Min:    {min(all_scores):.4f}")
        print(f"  Max:    {max(all_scores):.4f}")
        print(f"  Mean:   {np.mean(all_scores):.4f}")
        print(f"  Median: {np.median(all_scores):.4f}")
        print(f"  Std:    {np.std(all_scores):.4f}")
        print(f"\n  Suggested MIN_SCORE cutoffs:")
        for pct in [10, 25, 50]:
            val = np.percentile(all_scores, pct)
            print(f"    P{pct}: {val:.4f}")


if __name__ == "__main__":
    main()


Loaded 387 poems from '.\CSV\Diwan-Hamdan-WIP - Full_poems.csv'

Preparing text...

Generating embeddings using: local-minilm


ModuleNotFoundError: No module named 'sentence_transformers'

: 